In [1]:
from imblearn.under_sampling import RandomUnderSampler
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import pickle
from sklearn.model_selection import train_test_split

In [2]:
with open("C:\\Users\\Neela\\Documents\\GitHub\\EntityAspectLinking\\Experiment_trainsmall\\picklefiles\\baselinedataset_trainsmall.pkl", 'rb') as f:
    dataset = pickle.load(f)

In [3]:
with open("C:\\Users\\Neela\\Documents\\GitHub\\EntityAspectLinking\\Experiment_trainsmall\\picklefiles\\baselinedataset_test.pkl", 'rb') as f:
    test_dataset = pickle.load(f)

In [22]:
#Undersampling to meet class imbalances for class 0 and 1
X = dataset[:,:-1]
X_test = test_dataset[:,:-1]
sc = StandardScaler()
X = sc.fit_transform(X)
y = dataset[:,-1]
y_test = test_dataset[:, -1]
#undersample = RandomUnderSampler(sampling_strategy='majority')
#X, y = undersample.fit_resample(X, y)
#X_test, y_test = undersample.fit_resample(X_test, y_test)

In [23]:
count_neg = 0
for el in y:
    if el == 0:
        count_neg += 1

In [24]:
count_pos = y.shape[0] - count_neg

In [25]:
scale_ratio = count_neg / count_pos

In [27]:
#Train test split
#X_train,X_test,y_train,y_test = train_test_split(X, y,test_size = 0.3, random_state = 42)
X_train = X
y_train = y
dtrain = xgb.DMatrix(X_train, label=y_train)
dtest = xgb.DMatrix(X_test, label=y_test)

params = {
    'objective': 'binary:logistic',  
    'eval_metric': 'logloss',        
    'max_depth': 3,                  
    'learning_rate': 0.1,            
    'subsample': 0.8,                
    'colsample_bytree': 0.8,         
    'seed': 42,
    'scale_pos_weight':scale_ratio #class imbalance
} 
num_rounds = 100  
model = xgb.train(params, dtrain, num_rounds)
y_pred = model.predict(dtest)
y_pred_binary = [1 if pred > 0.5 else 0 for pred in y_pred]  # Convert probabilities to binary predictions

print("Accuracy:", accuracy_score(y_test, y_pred_binary))
print("\nClassification Report:\n", classification_report(y_test, y_pred_binary))

Accuracy: 0.5442955214122263

Classification Report:
               precision    recall  f1-score   support

         0.0       0.84      0.57      0.68     25623
         1.0       0.16      0.43      0.24      4967

    accuracy                           0.54     30590
   macro avg       0.50      0.50      0.46     30590
weighted avg       0.73      0.54      0.60     30590



In [28]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred_binary)

array([[14502, 11121],
       [ 2819,  2148]], dtype=int64)

In [41]:
import pandas as pd
test_id = pd.read_csv('Test_ID.csv')
test_id['Predicted Label'] = y_pred_binary
test_id['Predicted Probabilities'] = y_pred

In [43]:
test_id.to_csv('Predictions_trainsmall_XGboost.csv', index = False)

In [29]:
#Implementing Light GBM
import lightgbm as lgb
from sklearn.metrics import  roc_auc_score

# Create a LightGBM dataset
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

params = {
    'objective': 'binary',  # for binary classification 
    'metric': 'auc', # area under the curve
    'boosting_type': 'gbdt',    # traditional Gradient Boosting Decision Tree
    'num_leaves': 31,           # number of leaves in one tree
    'learning_rate': 0.05,      # learning rate
    'feature_fraction': 0.9,    # feature fraction
    'bagging_fraction': 0.8,    # bagging fraction
    'bagging_freq': 5,          # bagging frequency
    'verbose': 0,               # 0 for silent mode
    'scale_pos_weight': scale_ratio
}
# Train the model
num_round = 100  # Number of boosting rounds
bst = lgb.train(params, train_data, num_round, valid_sets = [test_data])
# Make predictions on the test set
y_pred_prob_lgb = bst.predict(X_test, num_iteration=bst.best_iteration)
y_pred_lgb = [1 if pred > 0.5 else 0 for pred in y_pred_prob_lgb]  # Convert probabilities to binary predictions

#Model evaluation 
accuracy = accuracy_score(y_test, y_pred_lgb)
roc_auc = roc_auc_score(y_test, y_pred_prob_lgb)

print(f'Accuracy on Test Set: {accuracy:.4f}')
print(f'ROC AUC on Test Set: {roc_auc:.4f}')
print("\nClassification Report:\n", classification_report(y_test, y_pred_lgb))


[LightGBM] [Warning] Auto-choosing col-wise multi-threading, the overhead of testing was 0.128924 seconds.
You can set `force_col_wise=true` to remove the overhead.
[1]	valid_0's auc: 0.468654
[2]	valid_0's auc: 0.477582
[3]	valid_0's auc: 0.487109
[4]	valid_0's auc: 0.498939
[5]	valid_0's auc: 0.506217
[6]	valid_0's auc: 0.503504
[7]	valid_0's auc: 0.502319
[8]	valid_0's auc: 0.505084
[9]	valid_0's auc: 0.508481
[10]	valid_0's auc: 0.510022
[11]	valid_0's auc: 0.509681
[12]	valid_0's auc: 0.51087
[13]	valid_0's auc: 0.511619
[14]	valid_0's auc: 0.511785
[15]	valid_0's auc: 0.511971
[16]	valid_0's auc: 0.511565
[17]	valid_0's auc: 0.511938
[18]	valid_0's auc: 0.50919
[19]	valid_0's auc: 0.507696
[20]	valid_0's auc: 0.505977
[21]	valid_0's auc: 0.504137
[22]	valid_0's auc: 0.504353
[23]	valid_0's auc: 0.501441
[24]	valid_0's auc: 0.5051
[25]	valid_0's auc: 0.505084
[26]	valid_0's auc: 0.502553
[27]	valid_0's auc: 0.501376
[28]	valid_0's auc: 0.499855
[29]	valid_0's auc: 0.499283
[30]	va

In [36]:
confusion_matrix(y_test, y_pred_lgb)

array([[14826, 10797],
       [ 2795,  2172]], dtype=int64)

In [46]:
import pandas as pd
test_id = pd.read_csv('Test_ID.csv')
test_id['Predicted Label'] = y_pred_lgb
test_id['Predicted Probabilities'] = y_pred_prob_lgb

In [49]:
test_id.to_csv('Predictions_trainsmall_LightGBM.csv', index = False)